# 03 — OTShield Benchmark & Evaluation
**Full system evaluation** — Cyber layer + Physical layer + Fusion

This notebook evaluates the complete OTShield detection pipeline on BATADAL dataset04, compares against the Zahoor et al. 2025 baseline, and generates presentation-ready visualizations.

**No models are retrained.** All artifacts are loaded from `models/`.

In [ ]:
# Cell 1 — Imports
import os, sys, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# Dark theme for presentation-ready plots
plt.rcParams.update({
    'figure.facecolor': '#080D12',
    'axes.facecolor':   '#0D1520',
    'axes.edgecolor':   '#2a3a4a',
    'axes.labelcolor':  '#8A9BB0',
    'text.color':       '#ffffff',
    'xtick.color':      '#8A9BB0',
    'ytick.color':      '#8A9BB0',
    'legend.facecolor': '#0D1520',
    'legend.edgecolor': '#2a3a4a',
    'grid.color':       '#1a2a3a',
    'font.size':        11,
})

print("Imports OK")

## 2 — Load Data

In [ ]:
# Cell 2 — Load dataset04
df = pd.read_csv('../data/BATADAL_dataset04.csv')
df.columns = [c.strip() for c in df.columns]

y_true = (df['ATT_FLAG'].values == 1).astype(int)

print(f"Dataset04: {len(df)} samples")
print(f"Normal: {(y_true == 0).sum()}, Attack: {(y_true == 1).sum()}")
print(f"Attack rate: {y_true.mean()*100:.1f}%")

## 3 — Load Trained Models
Load all saved artifacts directly for vectorized scoring (same logic as `cyber_layer.py` and `physical_layer.py` but without per-row SHAP overhead).

In [ ]:
# Cell 3 — Load model artifacts

# ── Cyber layer ──
with open('../models/feature_names_cyber.json') as f:
    CYBER_FEATURES = json.load(f)
with open('../models/optimized_thresholds_cyber.json') as f:
    CYBER_THRESHOLDS = json.load(f)

cyber_scaler = joblib.load('../models/scaler_cyber.pkl')
cyber_if     = joblib.load('../models/isolation_forest.pkl')
CYBER_BEST   = "isolation_forest"  # matches cyber_layer.py

# ── Physical layer ──
with open('../models/feature_names_physical.json') as f:
    PHYS_FEATURES = json.load(f)
with open('../models/optimized_thresholds_physical.json') as f:
    PHYS_THRESHOLDS = json.load(f)

phys_scaler = joblib.load('../models/scaler_physical.pkl')
phys_if     = joblib.load('../models/isolation_forest_physical.pkl')
phys_ocsvm  = joblib.load('../models/ocsvm_physical.pkl')
PHYS_BEST   = "isolation_forest"  # matches physical_layer.py

print(f"Cyber features:    {CYBER_FEATURES}")
print(f"Physical features: {PHYS_FEATURES}")
print(f"Cyber best model:    {CYBER_BEST}")
print(f"Physical best model: {PHYS_BEST}")
print(f"\nCyber thresholds:    {CYBER_THRESHOLDS}")
print(f"Physical thresholds: {PHYS_THRESHOLDS}")

# Quick sanity check
test_row = df[CYBER_FEATURES].iloc[0:1]
X_test = cyber_scaler.transform(test_row)
raw = cyber_if.decision_function(X_test)[0]
score = 0.5 + (CYBER_THRESHOLDS["isolation_forest"] - raw)
score = max(0.0, min(1.0, score)) * 100
print(f"\nSanity check — first row cyber score: {score:.1f}")

## 4 — Generate Predictions
Vectorized scoring using the same logic as `cyber_layer.py` and `physical_layer.py` — `score = 0.5 + (thresh - raw)`, scaled to 0-100.

In [ ]:
# Cell 4 — Vectorized scoring (same math as src/ layer files)

# ── Cyber scores ──
X_cyber = cyber_scaler.transform(df[CYBER_FEATURES])
cyber_raw = cyber_if.decision_function(X_cyber)
cyber_thresh = CYBER_THRESHOLDS["isolation_forest"]
cyber_scores = np.clip(0.5 + (cyber_thresh - cyber_raw), 0.0, 1.0) * 100

# ── Physical scores ──
X_phys = phys_scaler.transform(df[PHYS_FEATURES])
if PHYS_BEST == "ocsvm":
    phys_raw = phys_ocsvm.decision_function(X_phys)
    phys_thresh = PHYS_THRESHOLDS["ocsvm"]
else:
    phys_raw = phys_if.decision_function(X_phys)
    phys_thresh = PHYS_THRESHOLDS["isolation_forest"]
physical_scores = np.clip(0.5 + (phys_thresh - phys_raw), 0.0, 1.0) * 100

# ── Binary predictions ──
cyber_pred    = (cyber_scores > 50).astype(int)
physical_pred = (physical_scores > 50).astype(int)

# ── Fusion prediction ──
fusion_pred = np.where(
    (cyber_scores > 70) & (physical_scores > 70), 1,
    np.where(
        (cyber_scores > 50) | (physical_scores > 50), 1, 0
    )
)

# ── Fusion continuous score (weighted average for ROC) ──
fusion_scores = 0.5 * cyber_scores + 0.5 * physical_scores

print(f"Cyber predictions:    {cyber_pred.sum()} anomalies / {len(cyber_pred)} total")
print(f"Physical predictions: {physical_pred.sum()} anomalies / {len(physical_pred)} total")
print(f"Fusion predictions:   {fusion_pred.sum()} anomalies / {len(fusion_pred)} total")
print(f"\nScore ranges:")
print(f"  Cyber:    [{cyber_scores.min():.1f}, {cyber_scores.max():.1f}]")
print(f"  Physical: [{physical_scores.min():.1f}, {physical_scores.max():.1f}]")
print(f"  Fusion:   [{fusion_scores.min():.1f}, {fusion_scores.max():.1f}]")

## 5 — Compute Metrics

In [ ]:
# Cell 5 — Metrics comparison table

def compute_metrics(y_true, y_pred, scores):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, scores),
    }

m_cyber    = compute_metrics(y_true, cyber_pred,    cyber_scores)
m_physical = compute_metrics(y_true, physical_pred, physical_scores)
m_fusion   = compute_metrics(y_true, fusion_pred,   fusion_scores)

print("=" * 70)
print("OTShield — Full System Evaluation on BATADAL Dataset04")
print("=" * 70)
print(f"\n{'Layer':<16} | {'Precision':>9} | {'Recall':>6} | {'F1':>6} | {'ROC-AUC':>7}")
print("-" * 55)
for name, m in [("Cyber", m_cyber), ("Physical", m_physical), ("Fusion", m_fusion)]:
    print(f"{name:<16} | {m['Precision']:>9.3f} | {m['Recall']:>6.3f} | {m['F1']:>6.3f} | {m['ROC-AUC']:>7.3f}")
print(f"{'Zahoor et al.':<16} | {'---':>9} | {'---':>6} | {'0.835':>6} | {'0.891':>7}  <- baseline")
print()

# Highlight best
best_layer = max([("Cyber", m_cyber), ("Physical", m_physical), ("Fusion", m_fusion)],
                 key=lambda x: x[1]["F1"])
print(f"Best layer by F1: {best_layer[0]} — F1: {best_layer[1]['F1']:.3f}")

## 6 — Visualizations

In [ ]:
# Cell 6a — ROC Curves

fig, ax = plt.subplots(figsize=(8, 6))

for name, scores, color, ls in [
    ("Cyber",    cyber_scores,    '#4D9FFF', '-'),
    ("Physical", physical_scores, '#00CC88', '-'),
    ("Fusion",   fusion_scores,   '#FF1A1A', '-'),
]:
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    ax.plot(fpr, tpr, color=color, linewidth=2, linestyle=ls,
            label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], 'w--', alpha=0.2, linewidth=1)

# Zahoor baseline point
ax.scatter([1 - 0.891], [0.891], color='#FFB800', s=100, zorder=5,
           marker='*', label='Zahoor et al. 2025 (0.891)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('OTShield — ROC Curves by Layer', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6b — F1 Score Bar Chart

fig, ax = plt.subplots(figsize=(8, 5))

layers = ['Cyber', 'Physical', 'Fusion', 'Zahoor\net al. 2025']
f1_vals = [m_cyber['F1'], m_physical['F1'], m_fusion['F1'], 0.835]
colors  = ['#4D9FFF', '#00CC88', '#FF1A1A', '#FFB800']

bars = ax.bar(layers, f1_vals, color=colors, width=0.6, edgecolor='none')

# Add value labels on bars
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12,
            fontweight='bold', color='white')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('OTShield — F1 Score Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(f1_vals) * 1.2)
ax.axhline(y=0.835, color='#FFB800', linestyle='--', alpha=0.5, label='Baseline (0.835)')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6c — Confusion Matrices

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, preds, color) in zip(axes, [
    ("Cyber",    cyber_pred,    '#4D9FFF'),
    ("Physical", physical_pred, '#00CC88'),
    ("Fusion",   fusion_pred,   '#FF1A1A'),
]):
    cm = confusion_matrix(y_true, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Attack"])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} Layer', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.suptitle('OTShield — Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7 — Key Insights

### Multi-Modal Fusion Improves Detection Robustness

- **Cyber layer** detects network-level anomalies in SCADA command patterns (F_PU1, F_PU2, P_J280, P_J269 deviations)
- **Physical layer** detects process-level anomalies in tank levels and flow telemetry (L_T1, L_T2, L_T3 deviations)
- **Fusion** combines both perspectives — an attack that is subtle in one domain may be obvious in the other
- The combined system reduces false negatives by catching attacks that either layer alone would miss
- SHAP explainability pinpoints *which* sensor drove each alert, enabling targeted operator response

### Architecture Advantage
Unlike single-model approaches (Zahoor et al.), OTShield's tri-modal design allows independent layer upgrades. When audio and visual layers are activated, the fusion weights automatically adapt via the scoring engine in `fusion.py`.

## 8 — Demo Output Simulation
Simulates real-time dashboard alerts using `get_cyber_score()` and `get_physical_score()` on selected samples.

In [ ]:
# Cell 8 — Demo output simulation

# Add src/ to path so we can import the layer modules
sys.path.insert(0, os.path.abspath('../src'))
from cyber_layer import get_cyber_score
from physical_layer import get_physical_score

# Pick 3 representative samples: normal, attack (cyber-dominated), attack (physical-dominated)
normal_idx = np.where(y_true == 0)[0][100]
attack_indices = np.where(y_true == 1)[0]

# Find an attack where cyber scores high
attack_cyber_idx = attack_indices[np.argmax(cyber_scores[attack_indices])]
# Find an attack where physical scores high
attack_phys_idx = attack_indices[np.argmax(physical_scores[attack_indices])]

scenarios = [
    ("NORMAL OPERATION",    normal_idx),
    ("CYBER ATTACK",        attack_cyber_idx),
    ("PHYSICAL FAULT",      attack_phys_idx),
]

print("=" * 70)
print("OTShield — LIVE DEMO SIMULATION")
print("=" * 70)

for label, idx in scenarios:
    row = df.iloc[idx]
    sensor_dict = row.to_dict()

    c_score, c_expl, c_feat = get_cyber_score(sensor_dict)
    p_score, p_expl, p_feat = get_physical_score(sensor_dict)
    fusion = 0.5 * c_score + 0.5 * p_score

    # Classify
    if c_score > 70 and p_score > 70:
        alert = "CRITICAL SYSTEM ANOMALY"
        color = "RED"
    elif c_score > 50:
        alert = "CYBER ATTACK DETECTED"
        color = "BLUE"
    elif p_score > 50:
        alert = "PHYSICAL FAULT DETECTED"
        color = "GREEN"
    else:
        alert = "ALL SYSTEMS NOMINAL"
        color = "STEEL"

    print(f"\n--- [{label}] Sample #{idx} ---")
    print(f"  Cyber Score:    {c_score:>6.1f}  |  {c_expl}")
    print(f"  Physical Score: {p_score:>6.1f}  |  {p_expl}")
    print(f"  Fusion Score:   {fusion:>6.1f}")
    print(f"  Top features:   Cyber={c_feat}, Physical={p_feat}")
    print(f"  >>> ALERT: {alert} [{color}]")
    print(f"  Ground truth:   {'ATTACK' if y_true[idx] == 1 else 'NORMAL'}")

print("\n" + "=" * 70)
print("Demo simulation complete — ready for presentation")
print("=" * 70)